# 06 - Inference & Demo (3-Model Comparison)

Interactive inference comparing **all 3 trained models** (2014, 2015, 2016):

1. Load all 3 model checkpoints
2. Single text prediction — side-by-side comparison
3. Batch prediction on example reviews
4. Visual aspect highlighting per model
5. Custom text input

---

## 1. Setup

In [1]:
import subprocess, sys, os

def install_if_missing(package, pip_name=None):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or package])

install_if_missing('transformers')
install_if_missing('spacy')

import spacy
try:
    spacy.load('en_core_web_sm')
except OSError:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])

print('Dependencies ready.')

/home/thota23/miniconda3/envs/bash_ai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies ready.


In [2]:
PROJECT_ROOT = os.path.expanduser('~/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
from IPython.display import HTML, display
from src.inference import AspectSentimentPredictor, load_predictor

# Use CPU for inference demo — loads 3 models without GPU memory pressure
# Single-text inference is fast enough on CPU
device = torch.device('cpu')
print(f'Device: {device}')
print('Imports successful.')

Device: cpu
Imports successful.


## 2. Load All 3 Trained Models

In [3]:
import gc

# Load ALL available model checkpoints to CPU
predictors = {}  # year -> predictor
years = ['2014', '2015', '2016']

for year in years:
    ckpt = f'checkpoints/best_model_{year}.pt'
    if os.path.exists(ckpt):
        print(f'Loading {year} model from: {ckpt}')
        predictors[year] = load_predictor(year=year, device=device)
        print(f'  ✅ {year} model loaded successfully!')
    else:
        print(f'  ⚠️  No checkpoint found for {year} — skipping')

print(f'\n📊 Loaded {len(predictors)} model(s): {", ".join(predictors.keys())}')

gc.collect()

Loading 2014 model from: checkpoints/best_model_2014.pt


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 865.40it/s, Materializing param=layers.21.mlp_norm.weight]    
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/thota23/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews/src/inference.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_on

Loaded checkpoint: /home/thota23/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews/checkpoints/best_model_2014.pt
  Epoch: 6


  ✅ 2014 model loaded successfully!
Loading 2015 model from: checkpoints/best_model_2015.pt


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 850.14it/s, Materializing param=layers.21.mlp_norm.weight]    
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded checkpoint: /home/thota23/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews/checkpoints/best_model_2015.pt
  Epoch: 13
  ✅ 2015 model loaded successfully!
Loading 2016 model from: checkpoints/best_model_2016.pt


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 945.82it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded checkpoint: /home/thota23/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews/checkpoints/best_model_2016.pt
  Epoch: 12
  ✅ 2016 model loaded successfully!

📊 Loaded 3 model(s): 2014, 2015, 2016


605

## 3. Single Text — Compare All Models

In [4]:
text = "The pasta was absolutely delicious but the service was terribly slow."

print(f'Input: "{text}"')
print('=' * 70)

for year, pred in predictors.items():
    predictions = pred.predict(text)
    print(f'\n🔹 Model {year}:')
    if predictions:
        for p in predictions:
            emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐', 'conflict': '⚔️'}.get(p.sentiment, '❓')
            print(f'    {emoji} "{p.aspect}" → {p.sentiment} (conf: {p.confidence:.2f}, chars {p.start}:{p.end})')
    else:
        print('    No aspects detected.')
    print('-' * 70)

Input: "The pasta was absolutely delicious but the service was terribly slow."

🔹 Model 2014:
    😊 "pasta" → positive (conf: 1.00, chars 3:9)
    😞 "service" → negative (conf: 1.00, chars 42:50)
----------------------------------------------------------------------

🔹 Model 2015:
    😊 "pasta" → positive (conf: 0.99, chars 3:9)
    😞 "service" → negative (conf: 1.00, chars 42:50)
----------------------------------------------------------------------

🔹 Model 2016:
    😊 "pasta" → positive (conf: 0.94, chars 3:9)
    😞 "service" → negative (conf: 1.00, chars 42:50)
----------------------------------------------------------------------


In [5]:
# Visual HTML comparison — one panel per model
legend = '''
<div style="margin-top: 12px; font-size: 13px;">
  <b>Legend:</b>
  <span style="background: #27ae60; color: white; padding: 2px 8px; border-radius: 4px;">Positive</span>
  <span style="background: #e74c3c; color: white; padding: 2px 8px; border-radius: 4px;">Negative</span>
  <span style="background: #f39c12; color: white; padding: 2px 8px; border-radius: 4px;">Neutral</span>
  <span style="background: #8e44ad; color: white; padding: 2px 8px; border-radius: 4px;">Conflict</span>
</div>
'''

panels = []
for year, pred in predictors.items():
    predictions = pred.predict(text)
    highlighted = pred.get_highlighted_html(text, predictions)
    n_aspects = len(predictions)
    panels.append(
        f'<div style="flex: 1; min-width: 280px; padding: 12px; margin: 6px; '
        f'background: #f8f9fa; border-radius: 8px; border-left: 4px solid #3498db;">'
        f'<div style="font-weight: bold; color: #2c3e50; margin-bottom: 8px; font-size: 15px;">'
        f'🏷️ Model {year} ({n_aspects} aspect{"s" if n_aspects != 1 else ""})</div>'
        f'<div style="font-size: 16px; line-height: 2;">{highlighted}</div></div>'
    )

html = (
    f'<h3 style="color: #2c3e50;">🔍 Single Text Comparison</h3>'
    f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">{"".join(panels)}</div>'
    f'{legend}'
)
display(HTML(html))

## 4. Batch Prediction — All Models Side-by-Side

In [6]:
example_reviews = [
    "The spicy ramen was incredibly flavorful and the broth was rich.",
    "Terrible pizza with a soggy crust, but the drinks were excellent.",
    "Average food, nothing special about the ambiance either.",
    "The sushi here is the best I have ever had, fresh and perfectly seasoned.",
    "Long wait times and rude staff ruined an otherwise decent meal.",
    "Loved the cheesecake but the coffee was lukewarm and bitter.",
    "Great location with a beautiful patio, although prices are a bit high.",
    "The butter chicken was creamy and aromatic, paired perfectly with garlic naan.",
]

# Print text predictions for each review, grouped by model
for i, review in enumerate(example_reviews, 1):
    print(f'\n{"="*70}')
    print(f'[{i}] "{review}"')
    print(f'{"="*70}')
    
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        print(f'  📌 Model {year}: ', end='')
        if predictions:
            aspects_str = ', '.join(
                f'"{p.aspect}"→{p.sentiment}({p.confidence:.2f})' for p in predictions
            )
            print(aspects_str)
        else:
            print('No aspects detected')


[1] "The spicy ramen was incredibly flavorful and the broth was rich."
  📌 Model 2014: "spicy ramen"→positive(1.00), "broth"→positive(1.00)
  📌 Model 2015: "spicy"→positive(1.00), "broth"→positive(1.00)
  📌 Model 2016: "spicy ramen"→positive(1.00), "broth"→positive(1.00)

[2] "Terrible pizza with a soggy crust, but the drinks were excellent."
  📌 Model 2014: "pizza"→negative(1.00), "crust"→negative(1.00), "drinks"→positive(1.00)
  📌 Model 2015: "pizza"→negative(0.99), "drinks"→positive(1.00)
  📌 Model 2016: "pizza"→negative(0.95), "drinks"→positive(1.00)

[3] "Average food, nothing special about the ambiance either."
  📌 Model 2014: "food"→neutral(1.00), "am"→negative(0.58)
  📌 Model 2015: "food"→neutral(0.85), "am"→negative(0.67)
  📌 Model 2016: "food"→neutral(0.91), "am"→positive(0.60)

[4] "The sushi here is the best I have ever had, fresh and perfectly seasoned."
  📌 Model 2014: "sushi"→positive(1.00)
  📌 Model 2015: "s"→positive(0.98)
  📌 Model 2016: "sushi"→positive(1.00)

[5] "

In [7]:
# Visual HTML comparison for all reviews — each review shows all 3 models
all_blocks = []

for i, review in enumerate(example_reviews, 1):
    model_panels = []
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        highlighted = pred.get_highlighted_html(review, predictions)
        n_asp = len(predictions)
        model_panels.append(
            f'<div style="flex: 1; min-width: 250px; padding: 10px; '
            f'background: white; border-radius: 6px; border: 1px solid #dee2e6;">'
            f'<div style="font-weight: bold; font-size: 13px; color: #3498db; margin-bottom: 6px;">'
            f'Model {year} ({n_asp} aspect{"s" if n_asp != 1 else ""})</div>'
            f'<div style="font-size: 14px; line-height: 1.8;">{highlighted}</div></div>'
        )
    
    block = (
        f'<div style="margin: 16px 0; padding: 16px; background: #f8f9fa; '
        f'border-radius: 8px; border-left: 4px solid #2c3e50;">'
        f'<div style="font-weight: bold; margin-bottom: 10px; color: #2c3e50;">'
        f'[{i}] "{review}"</div>'
        f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">'
        f'{"".join(model_panels)}</div></div>'
    )
    all_blocks.append(block)

html = (
    '<h3 style="color: #2c3e50;">🍽️ Restaurant Review Analysis — 3-Model Comparison</h3>'
    + ''.join(all_blocks)
    + legend
)
display(HTML(html))

## 5. Prediction Details Table

In [8]:
import pandas as pd

# Collect all predictions from ALL models into a comparison table
rows = []
for review in example_reviews:
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        if predictions:
            for p in predictions:
                rows.append({
                    'Review': review[:45] + '...' if len(review) > 45 else review,
                    'Model': year,
                    'Aspect': p.aspect,
                    'Sentiment': p.sentiment,
                    'Confidence': f'{p.confidence:.3f}',
                    'Position': f'{p.start}:{p.end}',
                })
        else:
            rows.append({
                'Review': review[:45] + '...' if len(review) > 45 else review,
                'Model': year,
                'Aspect': '—',
                'Sentiment': '—',
                'Confidence': '—',
                'Position': '—',
            })

pred_df = pd.DataFrame(rows)

# Color-code by model
def color_model(val):
    colors = {
        '2014': 'background-color: #ebf5fb',
        '2015': 'background-color: #eafaf1',
        '2016': 'background-color: #fdf2e9',
    }
    return colors.get(val, '')

styled = (pred_df.style
    .set_caption('Prediction Details — All Models')
    .applymap(color_model, subset=['Model'])
)
display(styled)

/tmp/ipykernel_526629/4240101837.py:39: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = (pred_df.style


,Review,Model,Aspect,Sentiment,Confidence,Position
0,The spicy ramen was incredibly flavorful and ...,2014,spicy ramen,positive,1.000,3:15
1,The spicy ramen was incredibly flavorful and ...,2014,broth,positive,1.000,48:54
2,The spicy ramen was incredibly flavorful and ...,2015,spicy,positive,1.000,3:9
3,The spicy ramen was incredibly flavorful and ...,2015,broth,positive,1.000,48:54
4,The spicy ramen was incredibly flavorful and ...,2016,spicy ramen,positive,1.000,3:15
5,The spicy ramen was incredibly flavorful and ...,2016,broth,positive,1.000,48:54
6,"Terrible pizza with a soggy crust, but the dr...",2014,pizza,negative,1.000,8:14
7,"Terrible pizza with a soggy crust, but the dr...",2014,crust,negative,0.999,27:33
8,"Terrible pizza with a soggy crust, but the dr...",2014,drinks,positive,1.000,42:49
9,"Terrible pizza with a soggy crust, but the dr...",2015,pizza,negative,0.993,8:14


## 6. Agreement Analysis

How often do the 3 models agree on aspects and sentiments?

In [9]:
# Analyze where models agree/disagree
print('🔍 Model Agreement Analysis')
print('=' * 70)

for i, review in enumerate(example_reviews, 1):
    all_preds = {}
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        all_preds[year] = {p.aspect: p.sentiment for p in predictions}
    
    # Find all unique aspects across models
    all_aspects = set()
    for preds in all_preds.values():
        all_aspects.update(preds.keys())
    
    print(f'\n[{i}] "{review[:60]}..."' if len(review) > 60 else f'\n[{i}] "{review}"')
    
    if not all_aspects:
        print('  No aspects found by any model.')
        continue
    
    for aspect in sorted(all_aspects):
        sentiments = []
        for year in predictors.keys():
            s = all_preds[year].get(aspect, '—')
            sentiments.append(f'{year}:{s}')
        
        # Check agreement
        found_sentiments = [all_preds[y].get(aspect) for y in predictors if aspect in all_preds[y]]
        if len(set(found_sentiments)) == 1 and len(found_sentiments) == len(predictors):
            status = '✅ AGREE'
        elif len(found_sentiments) < len(predictors):
            status = '⚠️  PARTIAL'
        else:
            status = '❌ DISAGREE'
        
        print(f'  "{aspect}": {" | ".join(sentiments)}  [{status}]')

🔍 Model Agreement Analysis

[1] "The spicy ramen was incredibly flavorful and the broth was r..."
  "broth": 2014:positive | 2015:positive | 2016:positive  [✅ AGREE]
  "spicy": 2014:— | 2015:positive | 2016:—  [⚠️  PARTIAL]
  "spicy ramen": 2014:positive | 2015:— | 2016:positive  [⚠️  PARTIAL]

[2] "Terrible pizza with a soggy crust, but the drinks were excel..."
  "crust": 2014:negative | 2015:— | 2016:—  [⚠️  PARTIAL]
  "drinks": 2014:positive | 2015:positive | 2016:positive  [✅ AGREE]
  "pizza": 2014:negative | 2015:negative | 2016:negative  [✅ AGREE]

[3] "Average food, nothing special about the ambiance either."
  "am": 2014:negative | 2015:negative | 2016:positive  [❌ DISAGREE]
  "food": 2014:neutral | 2015:neutral | 2016:neutral  [✅ AGREE]

[4] "The sushi here is the best I have ever had, fresh and perfec..."
  "s": 2014:— | 2015:positive | 2016:—  [⚠️  PARTIAL]
  "sushi": 2014:positive | 2015:— | 2016:positive  [⚠️  PARTIAL]

[5] "Long wait times and rude staff ruined an otherw

## 7. Try Your Own Text

In [10]:
# Type your own review here!
custom_text = "The margherita pizza had a perfectly crispy crust but the toppings were bland."

print(f'Input: "{custom_text}"')
print('=' * 70)

panels = []
for year, pred in predictors.items():
    predictions = pred.predict(custom_text)
    highlighted = pred.get_highlighted_html(custom_text, predictions)
    n_asp = len(predictions)
    
    print(f'\n🔹 Model {year}:')
    if predictions:
        for p in predictions:
            emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐', 'conflict': '⚔️'}.get(p.sentiment, '❓')
            print(f'    {emoji} "{p.aspect}" → {p.sentiment} (conf: {p.confidence:.2f})')
    else:
        print('    No aspects detected.')
    
    panels.append(
        f'<div style="flex: 1; min-width: 280px; padding: 12px; margin: 6px; '
        f'background: #f8f9fa; border-radius: 8px; border-left: 4px solid #3498db;">'
        f'<div style="font-weight: bold; color: #2c3e50; margin-bottom: 8px;">'
        f'Model {year} ({n_asp} aspect{"s" if n_asp != 1 else ""})</div>'
        f'<div style="font-size: 16px; line-height: 2;">{highlighted}</div></div>'
    )

html = (
    f'<h3 style="color: #2c3e50;">🔍 Custom Text — 3-Model Comparison</h3>'
    f'<div style="display: flex; flex-wrap: wrap; gap: 8px;">{"".join(panels)}</div>'
    f'{legend}'
)
display(HTML(html))

Input: "The margherita pizza had a perfectly crispy crust but the toppings were bland."

🔹 Model 2014:
    😊 "margherita pizza" → positive (conf: 1.00)
    😊 "crust" → positive (conf: 1.00)
    😞 "toppings" → negative (conf: 0.53)

🔹 Model 2015:
    😞 "marg" → negative (conf: 0.43)
    😞 "ita pizza" → negative (conf: 0.72)
    😞 "topp" → negative (conf: 0.75)

🔹 Model 2016:
    😞 "marg" → negative (conf: 0.48)
    😞 "pizza" → negative (conf: 0.78)
    😐 "topp" → neutral (conf: 0.94)


## 8. Export Predictions

In [11]:
import json

# Export predictions from ALL models as JSON
export = []
for review in example_reviews:
    entry = {'text': review, 'predictions': {}}
    for year, pred in predictors.items():
        predictions = pred.predict(review)
        entry['predictions'][year] = [p.to_dict() for p in predictions]
    export.append(entry)

os.makedirs('outputs/results', exist_ok=True)
with open('outputs/results/inference_examples.json', 'w') as f:
    json.dump(export, f, indent=2)

total_aspects = sum(
    sum(len(e['predictions'][y]) for y in e['predictions'])
    for e in export
)
print(f'Predictions exported to: outputs/results/inference_examples.json')
print(f'Total reviews: {len(export)}')
print(f'Total aspects found (across all models): {total_aspects}')
print(f'Models included: {", ".join(predictors.keys())}')

Predictions exported to: outputs/results/inference_examples.json
Total reviews: 8
Total aspects found (across all models): 48
Models included: 2014, 2015, 2016


---

## Summary

| Component | Status |
|-----------|--------|
| Load all 3 models (2014, 2015, 2016) | ✅ |
| Side-by-side single text comparison | ✅ |
| Batch prediction across all models | ✅ |
| Visual HTML highlighting per model | ✅ |
| Model agreement analysis | ✅ |
| Custom text multi-model inference | ✅ |
| JSON export with all model predictions | ✅ |

> **Tip:** Compare which model finds more aspects and which has higher confidence scores to determine the strongest checkpoint.